In [1]:
import pandas as pd

In [89]:
df = pd.read_csv("../Datasets/npl_dataset.csv")
df.head(5)

,Unnamed: 0,match_id,inning,batting_team,bowling_team,ball_over,ball_result,bowler,batsman,non_striker,...,player_dismissed,dismissal_kind,fielder,batsman_runs,wide_runs,bye_runs,legbye_runs,noball_runs,extra_runs,total_runs
0,0,2024_1,1,Biratnagar Kings,Janakpur Bolts,0.1,0,Kishore Mahato,Lokesh Bam,Martin Guptill,...,NaN,NaN,NaN,0,0,0,0,0,0,0
1,1,2024_1,1,Biratnagar Kings,Janakpur Bolts,0.2,4,Kishore Mahato,Lokesh Bam,Martin Guptill,...,NaN,NaN,NaN,4,0,0,0,0,0,4
2,2,2024_1,1,Biratnagar Kings,Janakpur Bolts,0.3,0,Kishore Mahato,Lokesh Bam,Martin Guptill,...,NaN,NaN,NaN,0,0,0,0,0,0,0
3,3,2024_1,1,Biratnagar Kings,Janakpur Bolts,0.4,1w,Kishore Mahato,Lokesh Bam,Martin Guptill,...,NaN,NaN,NaN,0,1,0,0,0,1,1
4,4,2024_1,1,Biratnagar Kings,Janakpur Bolts,0.4,0,Kishore Mahato,Lokesh Bam,Martin Guptill,...,NaN,NaN,NaN,0,0,0,0,0,0,0


In [37]:
#For Wickets

possible_dismissal = df.groupby("dismissal_kind").size()
possible_dismissal.sort_values(ascending=False)

#Total Wickets by bowlers
dismissal_df = df[ (df["dismissal_kind"].notna()) & ( df["dismissal_kind"]
    .isin(["Bowled" , "bowled" ,
        "Stumped" , "stumped" ,
        "Caught" , "caught" ,
        "LBW" , "lbw" , 
        "Caught & Bowled" , "caught & bowled",
        "hit wicket"
       ]) ) ]
dismissal_df.head()
result = dismissal_df.groupby("bowler").size().reset_index(name="count").sort_values(by="count" , ascending=False).rename(columns={"bowler":"name" , "count":"value"}).head()
result

,name,value
88,Scott Kuggeleijn,27
42,Karan KC,25
91,Sher Malla,24
4,Abinash Bohara,24
84,Sandeep Lamichhane,23


In [130]:
#Most Runs Conceded
eligible_df = df[(df["legbye_runs"]==0) & (df["bye_runs"]==0) ]
total_runs = eligible_df.groupby("bowler")["total_runs"].sum().reset_index(name="runs_conceded").sort_values(by="runs_conceded" , ascending=False)
total_runs.head()

,bowler,runs_conceded
121,Sompal Kami,456
37,Dipendra Singh Airee,398
80,Nandan Yadav,383
6,Abinash Bohara,368
109,Sandeep Lamichhane,368


In [48]:
#For Best Bowling Average

#Total Runs Conceded By Individual Bowlers

eligible_df = df[(df["legbye_runs"]==0) & (df["bye_runs"]==0) ]
total_runs = eligible_df.groupby("bowler")["total_runs"].sum().reset_index(name="runs_conceded")

#Total Wickets by bowlers
dismissal_df = df[ (df["dismissal_kind"].notna()) & ( df["dismissal_kind"]
    .isin(["Bowled" , "bowled" ,
        "Stumped" , "stumped" ,
        "Caught" , "caught" ,
        "LBW" , "lbw" , 
        "Caught & Bowled" , "caught & bowled",
        "hit wicket"
       ]) ) ]
total_wickets = dismissal_df.groupby("bowler").size().reset_index(name="wickets")

#Combined df for total runs and total wickets
new_df = total_runs.merge(total_wickets , on="bowler")

#Calculating Average
new_df["average"] = round((new_df["runs_conceded"] / new_df["wickets"]) , 1)
result = new_df.sort_values(by="average" , ascending=True)
result.head()

,bowler,runs_conceded,wickets,average
83,Sandeep Chettri,5,1,5.0
15,Bikash Aagri,19,3,6.3
23,Dhananjaya Lakshan,67,10,6.7
66,Puneet Mehra,23,3,7.7
48,Lokesh Bam,8,1,8.0


In [60]:
#Total Strike Rate

#Total Ball Thrown
eligible_df = df[(df["wide_runs"]==0) & (df["noball_runs"]==0) ]
total_balls = eligible_df.groupby("bowler").size().reset_index(name="balls")

#Total Wickets Taken
dismissal_df = df[ (df["dismissal_kind"].notna()) & ( df["dismissal_kind"]
    .isin(["Bowled" , "bowled" ,
        "Stumped" , "stumped" ,
        "Caught" , "caught" ,
        "LBW" , "lbw" , 
        "Caught & Bowled" , "caught & bowled",
        "hit wicket"
       ]) ) ]
total_wickets = dismissal_df.groupby("bowler").size().reset_index(name="wickets")

#Combined df

combined_df = total_balls.merge(total_wickets , on="bowler")

#Calculating Strike Rate
combined_df["st_rate"] = round((combined_df["balls"] / combined_df["wickets"] ) , 1)
result = combined_df.sort_values(by="st_rate" , ascending = True).head()
result

,bowler,balls,wickets,st_rate
15,Bikash Aagri,18,3,6.0
48,Lokesh Bam,6,1,6.0
83,Sandeep Chettri,6,1,6.0
66,Puneet Mehra,19,3,6.3
24,Dinesh Kharel,7,1,7.0


In [87]:
#Best Bowling


#most wickets in inning
dismissal_df = df[ (df["dismissal_kind"].notna()) & ( df["dismissal_kind"]
    .isin(["Bowled" , "bowled" ,
        "Stumped" , "stumped" ,
        "Caught" , "caught" ,
        "LBW" , "lbw" , 
        "Caught & Bowled" , "caught & bowled",
        "hit wicket"
       ]) ) ]
most_wickets = dismissal_df.groupby(["match_id" , "bowler" , "batting_team"]).size().reset_index(name="wickets").sort_values(ascending=False , by="wickets").drop_duplicates(subset="match_id" , keep="first")

#total runs conceded by bowler having highest wicket in that match
eligible_df = df[(df["legbye_runs"]==0) & (df["bye_runs"]==0) ]
total_runs = eligible_df.groupby(["match_id" , "bowler"])["total_runs"].sum().reset_index(name="runs_conceded")

new_df = most_wickets.merge(total_runs , on=["match_id" , "bowler"]).head()
new_df


,match_id,bowler,batting_team,wickets,runs_conceded
0,2024_17,William Bosisto,Pokhara Avengers,6,28
1,2025_23,Shahab Alam,Pokhara Avengers,6,25
2,2024_2,Sohail Tanvir,Kathmandu Gurkhas,5,21
3,2024_14,Harsh Thaker,Lumbini Lions,5,23
4,2024_27,Scott Kuggeleijn,Pokhara Avengers,5,18


In [135]:
#Most 5 wicket Haul

#Most 5 Wickets in inning
dismissal_df = df[ (df["dismissal_kind"].notna()) & ( df["dismissal_kind"]
    .isin(["Bowled" , "bowled" ,
        "Stumped" , "stumped" ,
        "Caught" , "caught" ,
        "LBW" , "lbw" , 
        "Caught & Bowled" , "caught & bowled",
        "hit wicket"
       ]) ) ]
most_wickets = dismissal_df.groupby(["match_id" , "bowler" , "batting_team"]).size().reset_index(name="wickets").sort_values(ascending=False , by="wickets").drop_duplicates(subset="match_id" , keep="first")
most_wickets = most_wickets[most_wickets["wickets"]>=5]

result = most_wickets.groupby("bowler").size().reset_index(name="count")
result

,bowler,count
0,Harsh Thaker,1
1,Naren Saud,1
2,Ranjeet Kumar,1
3,Scott Kuggeleijn,1
4,Shahab Alam,1
5,Sohail Tanvir,1
6,William Bosisto,1


In [118]:
#Best Economy

#Total Runs Conceded
eligible_df = df[(df["legbye_runs"]==0) & (df["bye_runs"]==0) ]
total_runs = eligible_df.groupby("bowler")["total_runs"].sum().reset_index(name="runs_conceded")

#Total Overs used by bowlers
eligible_df = df[(df["wide_runs"]==0) & (df["noball_runs"]==0) ]
total_balls = eligible_df.groupby("bowler").size().reset_index(name="balls")
total_balls["overs"] = round((total_balls["balls"]/6) , 2)

#Calculting Economy
economy_df = total_runs.merge(total_balls , on="bowler")
economy_df["economy"] = round((economy_df["runs_conceded"] / economy_df["overs"]) , 2)
economy_df.sort_values(ascending=True , by="economy").head()

,bowler,runs_conceded,balls,overs,economy
4,Abhishek Gautam,78,96,16.0,4.88
13,Bas de Leede,20,24,4.0,5.00
108,Sandeep Chettri,5,6,1.0,5.00
79,Movin Subasingha,10,12,2.0,5.00
7,Alpesh Ramjani,21,24,4.0,5.25


In [124]:
#Most wide Balls

eligible_balls = df[df["wide_runs"]!=0]
eligible_balls.groupby("bowler").size().sort_values(ascending=False).reset_index(name="wides").head()


,bowler,wides
0,Scott Kuggeleijn,24
1,Kishore Mahato,19
2,Chris Sole,19
3,Pratish GC,17
4,Nandan Yadav,16


In [126]:
#Most Noballs
eligible_balls = df[df["noball_runs"]!=0]
eligible_balls.groupby("bowler").size().sort_values(ascending=False).reset_index(name="No Balls").head()

,bowler,No Balls
0,Amar Routela,4
1,Rijan Dhakal,3
2,Sompal Kami,3
3,Sohail Tanvir,3
4,James Neesham,2


In [127]:
eligible_balls = df[df["total_runs"]==0]
eligible_balls.groupby("bowler").size().sort_values(ascending=False).reset_index(name="Dot Balls").head()

,bowler,Dot Balls
0,Lalit Rajbanshi,179
1,Scott Kuggeleijn,176
2,Sandeep Lamichhane,156
3,Sohail Tanvir,154
4,Harmeet Singh,153
